# Exploratory Data Analysis (EDA) on Retail Sales Data
**Internship Project - Oasis Infobyte (Level 1)**  
**Author:** Aditya Halder (Data Analytics Intern)

---

### Project Overview
In this project, we perform a comprehensive Exploratory Data Analysis (EDA) on a retail sales dataset. The goal is to clean and preprocess the data, calculate descriptive statistics, analyze sales trends over time, identify customer demographic purchasing patterns, and extract actionable business insights to help the retail company make data-driven decisions.

### Dataset Selection Note
The internship project proposal linked to two datasets:
1. **Dataset 1**: Retail Sales Data (1,000 transaction records).
2. **Dataset 2**: McDonald's Menu Nutrition Data (260 food items details).

To satisfy the core requirement of performing an EDA on **Retail Sales Data**, this project is conducted using **Dataset 1**. Dataset 2 is kept in the directory structure under `Dataset/Dataset 2.csv` for structural alignment but is not part of this sales analysis.

### Key Concepts & Requirements
1. **Data Loading and Cleaning**: Load the dataset, handle missing values, check for duplicate records, and format data types.
2. **Descriptive Statistics**: Calculate summary metrics such as mean, median, mode, and standard deviation.
3. **Time Series Analysis**: Examine sales trends over weeks, months, and seasonal periods.
4. **Customer Demographics Analysis**: Study purchasing patterns based on age groups and gender.
5. **Product Category Analysis**: Analyze sales performance across different product categories.
6. **Advanced Customer Value Segmentation**: Implement Recency-Monetary (RM) value segmentation.
7. **Data Visualization**: Create charts (line charts, bar charts, donut charts, heatmaps) using Matplotlib and Seaborn.



## 1. Setup and Libraries
Importing required libraries for data manipulation, analysis, and visualization.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'

print("Libraries imported successfully!")


## 2. Loading the Dataset
Loading the retail sales dataset using Pandas. Note that we are using relative paths since the notebook is located in the `Notebook` folder.


In [ ]:
# Load the dataset
file_path = '../Dataset/Dataset 1.csv'
df = pd.read_csv(file_path)

# Display the first 5 records
df.head()


## 3. Data Cleaning and Preprocessing
Before conducting analysis, we must inspect the dataset's shape, data types, missing values, and check for duplicate rows to ensure data quality and integrity.


In [ ]:
# Inspect shape and basic information
print("Dataset Dimensions:", df.shape)
df.info()


### Check for Missing Values & Duplicates


In [ ]:
# Check for missing values in all columns
print("Missing values per column:\n", df.isnull().sum())

# Check for duplicate transactions
duplicates_count = df.duplicated().sum()
print(f"\nNumber of duplicate records: {duplicates_count}")


### Format Columns
We convert the `Date` column from object/string type to `datetime64` for time-series operations, and verify if the `Total Amount` is correct.


In [ ]:
# Convert Date to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Add month and year-month columns for time-series grouping
df['Year-Month'] = df['Date'].dt.to_period('M')
df['Day_of_Week'] = df['Date'].dt.day_name()

# Verify if Total Amount == Quantity * Price per Unit
is_correct = (df['Total Amount'] == df['Quantity'] * df['Price per Unit']).all()
print(f"Is Total Amount consistent across all records? {is_correct}")

df.head()


## 4. Descriptive Statistics
Generating basic statistical parameters (mean, median, mode, standard deviation, minimum, maximum) to summarize the distribution and central tendency of the numerical variables.


In [ ]:
# Descriptive stats for numerical columns
df.describe()


In [ ]:
# Value counts for categorical columns
print("Gender Breakdown:\n", df['Gender'].value_counts())
print("\nProduct Category Breakdown:\n", df['Product Category'].value_counts())


## 5. Time Series Analysis (Sales Trends over Time)
Let's analyze how sales fluctuate over time. We filter out the small tail of January 2024 to focus on the complete year of 2023 sales.


In [ ]:
# Filter 2023 sales for monthly trend analysis
df_2023 = df[df['Date'].dt.year == 2023].copy()
df_2023['Month'] = df_2023['Date'].dt.strftime('%m-%b')

# Group by Month and sum Total Sales
monthly_sales = df_2023.groupby('Month')['Total Amount'].sum().reset_index()

# Plot Monthly Sales Trend
plt.figure(figsize=(12, 6))
sns.lineplot(data=monthly_sales, x='Month', y='Total Amount', marker='o', color='#2b6cb0', linewidth=2.5, markersize=8)

plt.title('Monthly Sales Trend (2023)', pad=20, weight='bold', fontsize=15)
plt.xlabel('Month', labelpad=10)
plt.ylabel('Total Sales ($)', labelpad=10)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')

# Annotate peak and low
peak_idx = monthly_sales['Total Amount'].idxmax()
peak_month = monthly_sales.loc[peak_idx, 'Month']
peak_val = monthly_sales.loc[peak_idx, 'Total Amount']
plt.annotate(f'Peak: ${peak_val:,.0f}', xy=(peak_month, peak_val), xytext=(peak_month, peak_val + 3000),
             arrowprops=dict(facecolor='#2f855a', shrink=0.05, width=1.5, headwidth=6),
             horizontalalignment='center', weight='bold', color='#2f855a')

low_idx = monthly_sales['Total Amount'].idxmin()
low_month = monthly_sales.loc[low_idx, 'Month']
low_val = monthly_sales.loc[low_idx, 'Total Amount']
plt.annotate(f'Low: ${low_val:,.0f}', xy=(low_month, low_val), xytext=(low_month, low_val - 4000),
             arrowprops=dict(facecolor='#c53030', shrink=0.05, width=1.5, headwidth=6),
             horizontalalignment='center', weight='bold', color='#c53030')

# Save visualization
plt.tight_layout()
plt.savefig('../Visualizations/sales_trend.png', dpi=300)
plt.show()


## 6. Category-wise Sales Analysis
Next, we analyze sales across different product categories: Beauty, Clothing, and Electronics.


In [ ]:
# Group by Product Category
cat_sales = df.groupby('Product Category').agg(
    Transactions=('Transaction ID', 'count'),
    Total_Quantity=('Quantity', 'sum'),
    Total_Sales=('Total Amount', 'sum'),
    Average_Price=('Price per Unit', 'mean')
).reset_index()

cat_sales


### Visualizing Sales Distribution by Category (Donut Chart)


In [ ]:
# Donut Chart for Product Categories
plt.figure(figsize=(8, 8))
colors = ['#f6ad55', '#fc8181', '#63b3ed']
explode = (0.02, 0.02, 0.02)

plt.pie(cat_sales['Total_Sales'], explode=explode, labels=cat_sales['Product Category'], 
        autopct='%1.1f%%', startangle=140, colors=colors, 
        textprops={'fontsize': 12, 'weight': 'bold'},
        wedgeprops=dict(width=0.4, edgecolor='w'))

plt.title('Sales Distribution by Product Category', pad=20, weight='bold', fontsize=16)

# Save chart
plt.tight_layout()
plt.savefig('../Visualizations/category_sales.png', dpi=300)
plt.show()


## 7. Customer Demographics and Purchasing Analysis
Let's analyze who are our top-selling consumer segments by looking at **Gender** and **Age Groups**.


In [ ]:
# Group by Category and Gender
cat_gender = df.groupby(['Product Category', 'Gender'])['Total Amount'].sum().reset_index()

# Plot Sales by Category and Gender
plt.figure(figsize=(10, 6))
sns.barplot(data=cat_gender, x='Product Category', y='Total Amount', hue='Gender', palette=['#f687b3', '#4299e1'])
plt.title('Total Sales by Category and Gender', pad=20, weight='bold', fontsize=15)
plt.xlabel('Product Category', labelpad=10)
plt.ylabel('Total Sales ($)', labelpad=10)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.legend(title='Gender')

# Value labels on top of bars
for p in plt.gca().patches:
    height = p.get_height()
    if height > 0:
        plt.gca().annotate(f'${height:,.0f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='center',
                    xytext=(0, 9),
                    textcoords='offset points',
                    fontsize=9, weight='bold')

plt.ylim(0, cat_gender['Total Amount'].max() * 1.15)
plt.tight_layout()
plt.savefig('../Visualizations/top_products.png', dpi=300)
plt.show()


### Purchasing Behavior by Age Groups
Binning customers into specific age cohorts (18-25, 26-35, 36-45, 46-55, 56-65) to reveal spending habits by demographic age groups.


In [ ]:
# Bin customers by age
bins = [18, 25, 35, 45, 55, 65, 100]
labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
df['Age Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

# Group by Age Group
age_group_analysis = df.groupby('Age Group', observed=False).agg(
    Transactions=('Transaction ID', 'count'),
    Total_Quantity=('Quantity', 'sum'),
    Total_Sales=('Total Amount', 'sum'),
    Average_Spending=('Total Amount', 'mean')
).reset_index()

age_group_analysis


In [ ]:
# Plot Sales and Transactions by Age Group
fig, ax1 = plt.subplots(figsize=(10, 6))

color = '#4a5568'
ax1.set_xlabel('Age Group', labelpad=10)
ax1.set_ylabel('Number of Transactions', color=color)
sns.barplot(data=age_group_analysis, x='Age Group', y='Transactions', color='#a0aec0', ax=ax1, alpha=0.7)
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(False)

ax2 = ax1.twinx()  
color = '#e53e3e'
ax2.set_ylabel('Total Revenue ($)', color=color)
sns.lineplot(data=age_group_analysis, x='Age Group', y='Total_Sales', color=color, ax=ax2, marker='o', linewidth=2.5, markersize=8)
ax2.tick_params(axis='y', labelcolor=color)
ax2.yaxis.set_major_formatter('${x:,.0f}')
ax2.grid(False)

plt.title('Transactions vs. Revenue by Age Cohorts', pad=20, weight='bold', fontsize=15)
fig.tight_layout()
plt.show()


## 8. Advanced Customer Value Segmentation (Recency-Monetary Analysis)
In this section, we implement a **Recency-Monetary (RM)** customer value segmentation. 
*(Note: A check on the data indicates that there are exactly 1,000 unique Customer IDs for the 1,000 transaction records. Since Frequency is constant at 1, standard RFM collapses into a 2D Recency-Monetary grid).*

We define our scores as follows:
* **Recency (R)**: Days since last transaction relative to the maximum date in the dataset plus 1 day (`2024-01-02`).
  * `R_Score = 3` (Recent): $\le 120$ days ago.
  * `R_Score = 2` (Average): $120 < 	ext{days} \le 240$ ago.
  * `R_Score = 1` (Dormant): $> 240$ days ago.
* **Monetary (M)**: Transaction spending.
  * `M_Score = 3` (High spend): $\ge \$500$
  * `M_Score = 2` (Medium spend): $\$100 \le 	ext{spend} < \$500$
  * `M_Score = 1` (Low spend): $< \$100$



In [ ]:
# Set reference date
ref_date = df['Date'].max() + pd.Timedelta(days=1)

# Group by Customer ID to calculate Recency and Monetary metrics
customer_rfm = df.groupby('Customer ID').agg(
    Recency_Days=('Date', lambda x: (ref_date - x.max()).days),
    Monetary_Value=('Total Amount', 'sum')
).reset_index()

# Scoring helper functions
def r_score(days):
    if days <= 120: return 3
    elif days <= 240: return 2
    else: return 1

def m_score(spend):
    if spend >= 500: return 3
    elif spend >= 100: return 2
    else: return 1

customer_rfm['R_Score'] = customer_rfm['Recency_Days'].apply(r_score)
customer_rfm['M_Score'] = customer_rfm['Monetary_Value'].apply(m_score)

# Define behavioral segments
def segment_mapping(row):
    r, m = row['R_Score'], row['M_Score']
    if r == 3 and m == 3:
        return 'Champions'
    elif r == 3 and m < 3:
        return 'Active Prospects'
    elif r < 3 and m == 3:
        return 'At Risk'
    elif r == 2 and m < 3:
        return 'Regular Customers'
    else:
        return 'Lost Customers'

customer_rfm['Segment'] = customer_rfm.apply(segment_mapping, axis=1)

# Summarize customer segments
segment_summary = customer_rfm.groupby('Segment').agg(
    Customer_Count=('Customer ID', 'count'),
    Total_Sales=('Monetary_Value', 'sum'),
    Average_Spend=('Monetary_Value', 'mean')
).reset_index().sort_values(by='Total_Sales', ascending=False)

segment_summary


### Visualizing Customer Value Segments
Creating a dual-plot showing the number of customers in each segment and their total revenue contributions.


In [ ]:
# Colors mapping for consistency
colors_dict = {
    'At Risk': '#e53e3e',       # Red/Coral
    'Champions': '#319795',     # Teal
    'Lost Customers': '#718096',  # Slate Grey
    'Active Prospects': '#4299e1',# Blue
    'Regular Customers': '#ecc94b'# Yellow/Gold
}
colors_list = [colors_dict[seg] for seg in segment_summary['Segment']]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: Customer Count
sns.barplot(data=segment_summary, x='Customer_Count', y='Segment', palette=colors_list, ax=axes[0])
axes[0].set_title('Customer Count by Value Segment', pad=15, weight='bold', fontsize=13)
axes[0].set_xlabel('Number of Customers', labelpad=10)
axes[0].set_ylabel('Customer Segment', labelpad=10)

# Add value labels
for i, v in enumerate(segment_summary['Customer_Count']):
    axes[0].text(v + 3, i, str(v), va='center', fontweight='bold')

# Subplot 2: Total Revenue
sns.barplot(data=segment_summary, x='Total_Sales', y='Segment', palette=colors_list, ax=axes[1])
axes[1].set_title('Total Revenue Contribution by Segment', pad=15, weight='bold', fontsize=13)
axes[1].set_xlabel('Total Revenue ($)', labelpad=10)
axes[1].set_ylabel('')
axes[1].xaxis.set_major_formatter('${x:,.0f}')

# Add revenue labels
for i, v in enumerate(segment_summary['Total_Sales']):
    axes[1].text(v + 5000, i, f'${v:,.0f}', va='center', fontweight='bold')

plt.suptitle('Recency-Monetary (RM) Customer Value Segmentation Analysis', weight='bold', fontsize=16, y=0.98)
plt.tight_layout()
plt.savefig('../Visualizations/customer_segments.png', dpi=300)
plt.show()


## 9. Correlation Analysis
Understanding correlations between customer Age, Quantity purchased, Unit Price, and the Total Amount spent.


In [ ]:
# Correlation Heatmap
plt.figure(figsize=(8, 6))
numeric_cols = ['Age', 'Quantity', 'Price per Unit', 'Total Amount']
corr_matrix = df[numeric_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".3f", linewidths=.5, 
            square=True, cbar_kws={"shrink": .8}, annot_kws={'size': 12, 'weight': 'bold'})

plt.title('Correlation Matrix of Numerical Features', pad=20, weight='bold', fontsize=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('../Visualizations/heatmap.png', dpi=300)
plt.show()


## 10. Key Insights and Business Recommendations

Based on our exploratory data analysis, here are the main takeaways:
1. **Sales Seasonality & Trends**:
   - The retail store generated a total of **$456,000** in sales across 1,000 transactions in 2023.
   - Sales peaked in **May 2023 ($53,150)** and **October 2023 ($46,580)**.
   - There was a significant drop in **September 2023 ($23,620)**. Retailers should plan targeted marketing campaigns, discounts, or inventory restocking before this annual slump.
2. **Category Performance**:
   - **Electronics** generated the highest overall sales (**$156,905**), despite having slightly fewer transactions than Clothing. This is driven by high-ticket items.
   - **Clothing** saw the highest volume of transactions (351) and quantity sold (894 units).
   - **Beauty** was the lowest contributor in revenue ($143,515), representing an opportunity for category expansion or promotions.
3. **Advanced Customer Value Segmentation**:
   - **"At Risk" Revenue Concentration**: 242 customers are in the **At Risk** category (dormant high-value customers), yet they account for **$264,100 (57.9%) of total store revenue**. Winning back these clients is the highest priority business recommendation.
   - **Champions**: 108 customers are active high-value spenders, contributing **$125,000 (27.4%) of revenue**. They should be enrolled in early-access VIP campaigns.
   - **Prospects vs. Regulars**: Active Prospects (216 customers) and Regulars (201 customers) represent frequent but lower-ticket buyers. Upselling/bundling is recommended.
4. **Demographic Insights**:
   - **Gender Comparison**: Female customers spent slightly more overall (**$232,840**) compared to Male customers (**$223,160**). However, the average transaction size is almost identical (~$456).
   - **Gender & Product Affinity**: Female consumers spent significantly more on **Beauty** products ($83,860 vs. $59,655 for Males). Males spent more on **Electronics** ($84,650 vs. $72,255 for Females).
   - **Age Groups**: The **46-55** age group generated the most transactions (225) and revenue ($97,235), closely followed by the **26-35** and **36-45** brackets. The youngest group (**18-25**) had the highest average spend per transaction (**$501.01**).
5. **Correlation Results**:
   - **Total Amount** is highly correlated with **Price per Unit** (0.852) and moderately with **Quantity** (0.374).
   - Customer **Age** has zero correlation with purchasing behaviors (quantities, total spending, unit prices). This suggests product preferences and pricing limits are consistent across age brackets.

---
**End of Project**

